# polygon → MOC → shard → coincident waveforms

The waveform half of the minimal read stack. Same two libraries, same polygon,
same public store, zero credentials — but instead of the 3-D scatter, this
notebook does the **cell-level join**: one GEDI o18 footprint against the 2×2
ATL03 o19 cells beneath it, both reconstructed from their stored t-digests as
densities on a shared elevation axis.

Its sibling is [`hhdc_viewer.ipynb`](hhdc_viewer.ipynb), which takes the same
polygon and the same stores to a rotatable paired 3-D view and numpy tensors.
The two are separate notebooks on purpose: that one needs `%matplotlib widget`
for the 3-D view, this one needs `%matplotlib inline`, and the two backends
collide in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipywidgets
%matplotlib inline

import time

import moczarr as mz
from mortie import moc

# The drawing lives in viewers.py beside this notebook, so the cells below stay
# about the READ path. It is also where the one zagg import lives: moczarr
# imports the t-digest algebra rather than vendoring it (moczarr issue #19),
# which is what the `moczarr[zagg]` extra carries.
from viewers import BLOCK_ORDER, densest_shard, waveform_view

STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

ASIDE, GSIDE = 128, 64  # cells across an o12 block: 2**(19-12) and 2**(18-12)
N_BINS, RES = 256, 1.0  # shared z grid for the paired tensors

## One polygon in, covered shards out

The polygon below sits on SERC, the mid-Atlantic forest site the HHDC
diffusion papers are built on. Replace it with any area within California or a
NEON AOP site — it need not be a box — and the cell tests whether each store
actually covers what you asked for. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [
                        [-76.5750, 38.9000],
                        [-76.5450, 38.9000],
                        [-76.5450, 38.8780],
                        [-76.5600, 38.8720],
                        [-76.5750, 38.8800],
                        [-76.5750, 38.9000],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)

# Most GEDI granules -- same rule as hhdc_viewer and demo/06_paired.
SHARD = densest_shard(STORES["gedi"][0], shards, **S3)
print(f"{len(shards)} shards cover the polygon: {shards}\nworking {SHARD}")

## Paired tensors — one shard, both sensors

`read_tensors` yields one `(tensor, mask, (offset, gain), block)` per populated
o12 block. Reading both sensors on the same z grid is what makes the two
comparable; the blocks they share are the candidates for a join.

In [ ]:
stores = {name: mz.open_leaf(root, SHARD, **S3) for name, (root, _f) in STORES.items()}

t0 = time.perf_counter()
blocks = {
    name: {
        b[3]: b
        for b in mz.read_tensors(
            stores[name],
            field,
            n_bins=N_BINS,
            resolution=RES,
            block_order=BLOCK_ORDER,
            fit="degrade_resolution",
        )
    }
    for name, (_root, field) in STORES.items()
}
print(
    f"paired tensors in {time.perf_counter() - t0:.1f}s — "
    + ", ".join(f"{len(v)} {k} blocks" for k, v in blocks.items())
)

# Fold ATL03's o19 footprint down to GEDI's o18 grid, then keep the cells both
# sensors actually populate. SORTED by joint-cell count, densest first, so the
# first pick has real data on both sides.
pairs = []
for w, (gt, _gm, _gz, _) in blocks["gedi"].items():
    if w not in blocks["atl03"]:
        continue
    A2 = blocks["atl03"][w][0].sum(axis=2).reshape(GSIDE, 2, GSIDE, 2).sum(axis=(1, 3))
    G2 = gt.sum(axis=2)
    joint = (A2 > 0) & (G2 > 0)
    if joint.any():
        pairs.append((w, joint, A2, G2))
pairs.sort(key=lambda p: -int(p[1].sum()))

for w, j, A2, G2 in pairs[:8]:
    print(
        f"  {mz.morton_decimal(w)}  {int(j.sum()):4,} joint o18 cells   "
        f"{int(A2[j].sum()):9,} atl03 ph*   {int(G2[j].sum()):9,} gedi pe*"
    )
print("  * totals are from the TENSORS, so they count only what fell inside"
      " each block's z window (a few per cent under what the digests hold)")


## Coincident waveforms

A GEDI footprint cell (o18) covers exactly four ATL03 cells (o19). This reads all
four and merges their centroids (observations) into a single digest (waveform /
pdf), then compares that against GEDI's — so both sides describe the same patch of
ground at the same scale.

A centroid is one `(elevation, weight)` row. These stores are written with a
centroid budget far larger than any single cell fills, so nothing is merged away
at this scale: a centroid is one photon for ATL03 and one above-threshold
waveform sample for GEDI, and combining the four cells is a plain union of their
rows — nothing re-binned, re-compressed, or averaged.

Both sensors are read **straight from their stored digests** — no tensor in the
loop. The tensors above only decide which cells are worth comparing.

The slider orders those cells by joint signal strength, strongest first.
*Coincident* means the joint mask: both sensors present. *Strongest* needs a
little care. ATL03 counts photons; for GEDI the weight is an **estimate** of
detected photoelectrons — background-subtracted waveform counts scaled by a
named, versioned receiver gain (the published chain gives about 0.4 pe/count:
Sun et al. 2020, NASA NTRS
[20200001052](https://ntrs.nasa.gov/citations/20200001052); this store ships a
unit-gain placeholder, so its weights are *proportional* to photoelectrons rather
than calibrated to them). The two are not commensurate — GEDI's run two to three
orders of magnitude larger — so taking the smaller of the raw numbers would
always return ATL03's and rank by it alone. Each sensor is first divided by its
own maximum across the block's coincident cells, putting both on 0-1; the score
is the smaller of the two. A cell ranks high only when it is well populated for
**both** instruments.

In [ ]:
FIELDS = {name: field for name, (_root, field) in STORES.items()}
waveform_view(stores, FIELDS, blocks, pairs, SHARD)

Two libraries, one polygon — coverage, shards, paired tensors, and the
cell-level waveform join, anonymously against public S3.